In [9]:
import gymnasium as gym
import numpy as np

from stable_baselines3 import PPO

from stable_baselines3.common.callbacks import BaseCallback

from stable_baselines3.common.evaluation import evaluate_policy

class EpisodeRewardLogger(BaseCallback):

    def __init__(self):

        super().__init__()

        self.episode_rewards = []

        self.current_reward = 0.0

    def _on_step(self):

        self.current_reward += self.locals[
            "rewards"
        ][0]

        if self.locals[
            "dones"
        ][0]:

            self.episode_rewards.append(
                self.current_reward
            )

            self.current_reward = 0.0

        return True


def train_ppo():

    env = gym.make(
        "CartPole-v1"
    )

    callback = EpisodeRewardLogger()

    model = PPO(
        "MlpPolicy",
        env,
        verbose=0,

        learning_rate=3e-4,

        n_steps=2048,

        batch_size=64,

        gamma=0.99
    )

    model.learn(
        total_timesteps=50000,
        callback=callback
    )

    mean_reward, std_reward = evaluate_policy(
        model,
        env,
        n_eval_episodes=10
    )

    print(
        f"PPO evaluation: "
        f"{mean_reward:.1f} +/- {std_reward:.1f}"
    )

    np.save(
        "ppo_returns.npy",
        np.array(
            callback.episode_rewards
        )
    )

    env.close()

    return callback.episode_rewards


ppo_returns = train_ppo()

PPO evaluation: 500.0 +/- 0.0
